# Structure-aware prompting on HANS: run every experiment

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When the last cell finishes, `results.zip` downloads automatically.

Rough time estimate on a T4: 15–30 min for Qwen2.5-3B-Instruct (including the model download), about 10 min for the optional Qwen2.5-1.5B-Instruct run, and 2 min for the optional greedy check.
Inference resumes where it stopped if a cell is interrupted and re-run.

`results.zip` contains, for each model:
raw model responses (`predictions.jsonl`, `pilot_predictions.jsonl`),
run metadata with library versions, GPU and decoding settings (`run_meta_*.json`),
every table, the figure, the example sheets and the annotation sheet (`analysis/`),
plus `env/` (`pip freeze`, `nvidia-smi`) for the whole session.

In [ ]:
# 1. Check that a GPU is attached
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then Run all again."
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code (public GitHub repo). Leave REPO_URL empty to upload structprompt-hans.zip instead.
REPO_URL = "https://github.com/haeeyoung/structprompt-hans.git"

import os
if not os.path.exists("/content/structprompt-hans"):
    if REPO_URL:
        !git clone -q {REPO_URL} /content/structprompt-hans
    else:
        from google.colab import files
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        !mkdir -p /content/structprompt-hans && unzip -q -o "{zip_name}" -d /content/structprompt-hans
%cd /content/structprompt-hans
!git log -1 --format="code version: %h (%ad)" 2>/dev/null || true

In [ ]:
# 3. Install pinned library versions (torch is Colab's preinstalled build; its version is logged)
!pip install -q -r requirements.txt

# Record the exact environment; it goes into results.zip
!mkdir -p results/env && pip freeze > results/env/pip_freeze.txt && nvidia-smi > results/env/nvidia_smi.txt && python --version > results/env/python_version.txt
!cat results/env/python_version.txt && grep -iE "^(torch|transformers|accelerate)==" results/env/pip_freeze.txt

In [ ]:
# 4. Sanity checks: label parser, and the 300-pair sample re-derived from HANS (pinned commit, seed 42)
!python tests/test_parse.py
!python sample_hans.py --verify

In [ ]:
# 5. Instruction token lengths (P4 must be approximately length-matched to P3)
!python check_prompts.py --tag qwen2.5-3b-instruct

In [ ]:
# 6. Pilot: 12 HANS *training* pairs x 4 prompts, only to check the answer format
!python run_inference.py --tag qwen2.5-3b-instruct --split pilot

import json
for line in open("results/qwen2.5-3b-instruct/pilot_predictions.jsonl", encoding="utf-8"):
    r = json.loads(line)
    print(f"{r['condition']} gold={r['gold']:<15} pred={str(r['pred']):<15} | {r['output'][:160]!r}")

In [ ]:
# 7. Main run: 300 evaluation pairs x 4 prompts = 1,200 predictions
!python run_inference.py --tag qwen2.5-3b-instruct --split eval

In [ ]:
# 8. Analysis: tables, bootstrap CIs, McNemar tests, figure, example sheets
!python analyze.py --tag qwen2.5-3b-instruct

from IPython.display import Image, display
display(Image("results/qwen2.5-3b-instruct/analysis/fig_accuracy.png"))

## Optional robustness check: Qwen2.5-1.5B-Instruct

Same 300 pairs, same prompts, same decoding. Set `RUN_1_5B = False` to skip.

In [ ]:
RUN_1_5B = True

if RUN_1_5B:
    !python run_inference.py --tag qwen2.5-1.5b-instruct --split pilot
    !python run_inference.py --tag qwen2.5-1.5b-instruct --split eval
    !python analyze.py --tag qwen2.5-1.5b-instruct

## Optional: greedy-decoding check

Confirms that `generate()` with our settings produces exactly the tokens of a hand-written argmax loop
(no repetition penalty, no sampling), on 8 pilot prompts with the main model in float16.
Also reports whether batching changed any output. Writes `results/qwen2.5-3b-instruct/greedy_check.json`.
Set `RUN_GREEDY_CHECK = False` to skip.

In [ ]:
RUN_GREEDY_CHECK = True

if RUN_GREEDY_CHECK:
    !python tests/check_greedy.py --tag qwen2.5-3b-instruct

In [ ]:
# 10. Check that every expected file exists, then package and download results.zip
import os

PER_MODEL = ["predictions.jsonl", "run_meta_eval.json", "pilot_predictions.jsonl", "run_meta_pilot.json",
             "analysis/summary.md", "analysis/summary.json", "analysis/item_level.csv",
             "analysis/table_main.tex", "analysis/table_tests.tex", "analysis/table_subcase.tex",
             "analysis/fig_accuracy.pdf", "analysis/fig_accuracy.png",
             "analysis/flips_P2_P3.md", "analysis/random_examples.md", "analysis/annotation_sheet_P3.csv"]
tags = ["qwen2.5-3b-instruct"] + (["qwen2.5-1.5b-instruct"] if globals().get("RUN_1_5B") else [])
expected = ["results/env/pip_freeze.txt", "results/env/nvidia_smi.txt",
            "results/qwen2.5-3b-instruct/prompt_lengths.json"]
expected += [f"results/{t}/{f}" for t in tags for f in PER_MODEL]
if globals().get("RUN_GREEDY_CHECK"):
    expected.append("results/qwen2.5-3b-instruct/greedy_check.json")

for t in tags:
    path = f"results/{t}/predictions.jsonl"
    n = sum(1 for _ in open(path, encoding="utf-8")) if os.path.exists(path) else 0
    print(f"{t}: {n} predictions (expected 1200)")
missing = [p for p in expected if not os.path.exists(p)]
print("all expected files present" if not missing else "MISSING (scroll up for the error):\n  " + "\n  ".join(missing))

!rm -f results.zip && zip -qr results.zip results -x "results/_fake/*" "results/_smoke/*"
!unzip -l results.zip
from google.colab import files
files.download("results.zip")